In [219]:
from pyspark.sql import functions as f
from pyspark.sql.functions import *
from pyspark.sql import Window
import logging

In [220]:
bi_contrato = spark.read.parquet ('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BI_CONTRATO/')

base_zeus = spark.read.parquet ('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BASE_ZEUS/')

titulos_vertia = spark.read.parquet ('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BI_TITULOS_VERTIA/')

bi_originais = spark.read.parquet ('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BI_ORIGINAIS/')

bi_acordos_promessas = spark.read.parquet ('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BI_ACORDOS_PROMESSAS/')

bi_parcelas = spark.read.parquet ('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BI_PARCELAS/')

polos_proprios_e_parceiros = spark.read.parquet('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/POLOS_PARCEIROS_E_PROPRIOS/')

ciclo_remat = spark.read.parquet('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/CICLO_REMAT/')

In [221]:
#codigo novo
# ORIGINAIS GROUP 
def join_originais_group(bi_originais):
    win = Window.partitionBy(f.col("COD_TITULO")).orderBy(f.col("DAT_PROCESSAMENTO_ORI").desc())  
    df_bi_originais = (bi_originais
                            .withColumn("row", f.row_number().over(win))
                            .filter((f.col("row") == '1'))) 
    
    df_bi_originais = (df_bi_originais.select(f.col('COD_TITULO'),
                                              f.col('COD_CONTRATO_ORI'),
                                              f.col('COD_PARCELAMENTO_ORI'),
                                              f.col('NUM_MES_COMPETENCIA'),
                                              f.col('NUM_ANO_COMPETENCIA'),
                                              f.col('NUM_PARCELA'),
                                              f.col('IND_NATUREZA_REMATRICULA'),
                                              f.col('STA_TITULO'),
                                              f.col('DSC_NATUREZA_TITULO'),
                                              f.col('NOM_NATUREZA_TITULO'),
                                              f.col('DAT_VENCIMENTO')
                                             ))
    
    return df_bi_originais
    
    
    
# ACORDOS GROUP 
def join_acordos_group(bi_acordos_promessas):    
    win = Window.partitionBy(f.col("COD_PARCELAMENTO")).orderBy(f.col("DAT_PROCESSAMENTO").desc())   
    df_bi_acordos_promessas = (bi_acordos_promessas
                                    .withColumn("row", f.row_number()
                                    .over(win))
                                    .filter((f.col("row") == '1')))  
    
    df_bi_acordos_promessas = df_bi_acordos_promessas.select(f.col("DAT_PROCESSAMENTO"),
                                                             f.col("TIP_TP_MATRICULA"),
                                                             f.col("COD_CONTRATO"),
                                                             f.col("DSC_MARCA_CONTRATO"),
                                                             f.col("COD_CLIENTE"),
                                                             f.col("NOM_NM_CLIENTE"),
                                                             f.col("COD_PARCELAMENTO"),
                                                             f.col("DAT_CRIACAO_PARCELAMENTO"),
                                                             f.col("TIP_TP_PARCELAMENTO"),
                                                             f.col("NUM_QTDE_PARCELAS"),
                                                             f.col("VLR_TOTAL"),
                                                             f.col("VLR_PRINCIPAL"),
                                                             f.col("VLR_DESC_PRINCIPAL"),
                                                             f.col("VLR_MULTA"),
                                                             f.col("VLR_DESC_MULTA"),
                                                             f.col("VLR_JUROS_MORA"),
                                                             f.col("VLR_JUROS_FINANC"),
                                                             f.col("VLR_DESC_JUROS_FINANC"),
                                                             f.col("VLR_CORRECAO_MONETARIA"),
                                                             f.col("VLR_DESC_CORRECAO_MONETARIA"),
                                                             f.col("VLR_TAXA_NEGOCIACAO"),
                                                             f.col("VLR_DESC_TAXA_NEGOCIACAO"),
                                                             f.col("DSC_GRUPO_USUARIO"),
                                                             f.col("DSC_ATENDIMENTO"),
                                                             f.col("NOM_US_CRIACAO_PARCELAMENTO"),
                                                             f.col("STA_CANCELAMENTO")
                                                            )

    return df_bi_acordos_promessas


# PARCELAS GROUP 
def join_parcelas_group(bi_parcelas):    
    win = Window.partitionBy(f.col("COD_PARCELAMENTO_PAR"),f.col("COD_PARCELA")).orderBy(f.col("DAT_PROCESSAMENTO").desc())   
    df_bi_parcelas = (bi_parcelas
                   .withColumn("row", f.row_number().over(win))
                   .filter((f.col("row") == '1')))   
    
    df_bi_parcelas = df_bi_parcelas.select(f.col('COD_PARCELAMENTO_PAR'),
                                           f.col('COD_PARCELA'),
                                           f.col('COD_CONTRATO'),
                                           f.col('COD_TITULO'),
                                           f.col('NUM_TITULO'),
                                           f.col('DAT_PAGAMENTO'),
                                           f.col('VLR_PAGAMENTO'),
                                           f.col('DAT_VENCIMENTO'),
                                           f.col('VLR_PARCELA')
                                          )
    
    return df_bi_parcelas


In [222]:
df_acordos_promessas = join_acordos_group(bi_acordos_promessas)
df_parcelas = join_parcelas_group(bi_parcelas)
df_originais = join_originais_group(bi_originais)

In [223]:
def join_parcelas_acordos(df_acordos_promessas, df_parcelas):
    df_acordos_promessas = (df_acordos_promessas.join(df_parcelas,
                                  df_acordos_promessas['COD_PARCELAMENTO'] == df_parcelas['COD_PARCELAMENTO_PAR'], "left"))
      
    
    return (df_acordos_promessas
            .drop(df_parcelas['COD_CONTRATO'])
            .drop(df_acordos_promessas['DAT_PROCESSAMENTO'])
            .drop(df_parcelas['COD_PARCELAMENTO_PAR']) 
            .drop(df_acordos_promessas['COD_CLIENTE'])
           )

In [224]:
df_acordos_promessas = join_parcelas_acordos(df_acordos_promessas, df_parcelas)

In [177]:
df_acordos_promessas.count()

5313413

In [225]:
df_acordos_promessas = (df_acordos_promessas
                            .withColumn("TIP_PARCELA",
                            f.when((f.col('COD_PARCELA') == "1") & (f.col("VLR_PAGAMENTO").isNull()), "PROMESSA")
                            .when((f.col('COD_PARCELA') == "1") & (~(f.col("VLR_PAGAMENTO").isNull())), "ACORDO")
                            .when((~(f.col('COD_PARCELA') == "1")) & ((f.col("VLR_PAGAMENTO").isNull())) & (f.to_date(date_format(f.col('DAT_VENCIMENTO'),"yyyy-MM-dd"),"yyyy-MM-dd") > f.current_date()), "COLCHAO")
                            .when((~(f.col('COD_PARCELA') == "1")) & ((f.col("VLR_PAGAMENTO").isNull())) & (f.to_date(date_format(f.col('DAT_VENCIMENTO'),"yyyy-MM-dd"),"yyyy-MM-dd") < f.current_date()), "COLCHAO EM ATRASO")
                            .when((~(f.col('COD_PARCELA') == "1")) & (~(f.col("VLR_PAGAMENTO").isNull())) , "COLCHAO PAGO")
                        ))

In [226]:
df_acordos_promessas =  (df_acordos_promessas.withColumn("NOM_SISTEMA", f.when(f.substring(f.col("COD_CONTRATO"), 1, 6) == f.lit("OLIMPO"),
                                                     f.lit("OLIMPO"))
                                            .otherwise(
                                                f.concat(f.lit("COLABORAR"))
                                            )))

In [232]:
## EMPACOTAMENTO
def join_empac(df_acordos_promessas, df_originais, bi_ciclo_remat): 
    df_empac = (df_originais.filter(
                                (f.col('NOM_NATUREZA_TITULO') == 'MENSALIDADE') &
                                (f.col('IND_NATUREZA_REMATRICULA') == 'S' ) &
                                (f.col('NUM_ANO_COMPETENCIA') == f.year(f.current_date()))
                    ))
    
    df_empac = df_empac.select(f.col('COD_PARCELAMENTO_ORI'),f.col('NOM_NATUREZA_TITULO'),f.col('NUM_ANO_COMPETENCIA'),f.col('IND_NATUREZA_REMATRICULA'))
    
    df_acordos_promessas = (df_acordos_promessas.join(df_empac,
                                  df_acordos_promessas['COD_PARCELAMENTO'] == df_empac['COD_PARCELAMENTO_ORI'], "left"))

    df_acordos_promessas = (df_acordos_promessas.join(bi_ciclo_remat,
                                  df_acordos_promessas['NOM_SISTEMA'] == bi_ciclo_remat['NOM_SISTEMA'], "left"))
    
    df_acordos_promessas = (df_acordos_promessas
                            .withColumn("CHAVE",
                            f.concat(f.concat(f.col('COD_PARCELA'),f.col('COD_PARCELAMENTO')),f.col('COD_CONTRATO'))
                  ))
    
    # CODIGO DE EMPAC
    df_acordos_promessas = (df_acordos_promessas
                            .withColumn("COD_EMPAC",
                            f.when((f.to_date(date_format(f.col('DAT_CRIACAO_PARCELAMENTO'),"yyyy-MM-dd"),"yyyy-MM-dd") >= f.to_date(date_format(f.col('DAT_INICIO'),"yyyy-MM-dd"),"yyyy-MM-dd")) & (f.to_date(date_format(f.col('DAT_CRIACAO_PARCELAMENTO'),"yyyy-MM-dd"),"yyyy-MM-dd") <= f.to_date(date_format(f.col('DAT_INICIO'),"yyyy-MM-dd"),"yyyy-MM-dd")) & (f.col('IND_NATUREZA_REMATRICULA') == 'S' ) & (f.col('NUM_ANO_COMPETENCIA') == f.year(f.current_date())), f.col('NOM_CICLO'))
                  ))    
    
    win = Window.partitionBy(f.col("CHAVE")).orderBy(f.col("CHAVE").desc())  
    df_acordos_promessas = (df_acordos_promessas 
                          .withColumn("row", f.row_number().over(win))
                          .filter((f.col("row") == '1')))
     
    return (df_acordos_promessas
           #.drop(df_empac['DAT_VENCIMENTO'])
            #.drop(df_empac['COD_TITULO'])
            #.drop(df_empac['COD_CLIENTE_ORI'])
            #.drop(df_empac['COD_CONTRATO_ORI'])
            .drop(df_empac['COD_PARCELAMENTO_ORI'])
            .drop(bi_ciclo_remat['NOM_SISTEMA'])
            .drop(bi_ciclo_remat['NUM_ANO'])
            .drop(bi_ciclo_remat['NUM_SEMESTRE'])
            .drop(bi_ciclo_remat['execution_ts'])
            #.drop(bi_originais['IND_NATUREZA_REMATRICULA'])
            #.drop(df_empac['DAT_PROCESSAMENTO_ORI'])
            #.drop(df_empac['NOM_GRUPO_TELA_REMAT'])
            #.drop(bi_originais['NOM_NATUREZA_TITULO'])
            #.drop(df_empac['DSC_NATUREZA_TITULO'])
            #.drop(df_empac['VLR_LIQUIDO'])
            #.drop(df_empac['NUM_CONTADOR_TELA_REMAT'])
            #.drop(ciclo_remat['NOM_CICLO'])
            #.drop(bi_ciclo_remat['NOM_SISTEMA'])
            #.drop(bi_ciclo_remat['NUM_ANO'])
            #.drop(bi_ciclo_remat['NUM_SEMESTRE'])
            #.drop(ciclo_remat['DAT_INICIO'])
            #.drop(ciclo_remat['DAT_FIM'])
            )


In [233]:
bi_ciclo_remat = ciclo_remat

In [234]:
df_acordos_promessas = join_empac(df_acordos_promessas, df_originais, bi_ciclo_remat)

In [235]:
df_acordos_promessas.count()

5313337

In [236]:
#-- DIAS ATRASO
def join_dias_atraso(df_acordos_promessas, df_originais):
    
    df_originais = df_originais.drop(f.col('COD_TITULO')) 
    
    #group by
    bi_originais =  (df_originais.groupBy('COD_PARCELAMENTO_ORI')
                                .agg((f.min(f.to_date(date_format(f.col('DAT_VENCIMENTO'),"yyyy-MM-dd"),"yyyy-MM-dd"))).alias('DAT_MENOR_VENCIMENTO'))) 
    
    df_acordos_promessas = (df_acordos_promessas.join(bi_originais,
                                  df_acordos_promessas['COD_PARCELAMENTO'] == bi_originais['COD_PARCELAMENTO_ORI'], "left"))
    
    return (df_acordos_promessas
            .drop(bi_originais['COD_PARCELAMENTO_ORI'])
           )

In [237]:
bi_contratos = bi_contrato

In [238]:
df_acordos_promessas = join_dias_atraso(df_acordos_promessas, df_originais)

In [239]:
df_acordos_promessas.count()

5313337

In [240]:
def join_contratos(df_acordos_promessas, bi_contratos):
    
    df_acordos_promessas = (df_acordos_promessas.join(bi_contratos,
                                  df_acordos_promessas["COD_CONTRATO"] == bi_contratos["COD_CONTRATO"], "left"))


    df_acordos_promessas =  (df_acordos_promessas.withColumn("COD_CPFCGC_PES", f.when(f.col("TIP_MATRICULA") == "PRESENCIAL",
                            f.concat(
                                   f.lpad(f.col("COD_FILIAL"),3,'0'), f.lit(""), f.substring(f.col('COD_CONTRATO_ORIGEM'), 2, 15)
                                    )
                                                                                )
                            .otherwise(
                               f.concat(f.lit("EAD"),f.lit(""), f.col("COD_CONTRATO_ORIGEM"))
                                      )
                                                        )
                        )

    
    return (df_acordos_promessas
            .drop(bi_contratos['COD_CONTRATO'])
            .drop(bi_contratos['DAT_PROCESSAMENTO'])
            .drop(bi_contratos['COD_REGISTRO'])
            .drop(bi_contratos['COD_CLIENTE'])
            .drop(bi_contratos['DSC_SITUACAO_CONTRATO'])
            .drop(bi_contratos['execution_ts'])
           )
           

In [241]:
df_acordos_promessas = join_contratos(df_acordos_promessas, bi_contratos)

In [242]:
def join_zeus_atual(df_acordos_promessas, bi_zeus):  
    
    df_zeus = (bi_zeus.filter(~(f.col('TIP_ALUNO')== "ACORDO RAID")))
    df_zeus = (df_zeus.select(f.col('COD_RA_ZEUS'),f.col('STA_OPERACAO_ZEUS'),f.col('TIP_CURSO_ZEUS'),f.col('COD_SITUACAO_ACADEMICA'),f.col('DSC_SITUACAO_ACADEMICA')))
    
    df_acordos_atual = (df_acordos_promessas.filter((f.year(f.col('DAT_CRIACAO_PARCELAMENTO')) == f.year(f.current_date())) & (f.month(f.col('DAT_CRIACAO_PARCELAMENTO')) == f.month(f.current_date()))))
    df_acordos_atual = (df_acordos_atual.join(df_zeus,
                                  df_acordos_atual['COD_CPFCGC_PES'] == df_zeus['COD_RA_ZEUS'], "inner"))
    
    df_acordos_atual = df_acordos_atual.select(f.col('COD_PARCELAMENTO'),f.col('STA_OPERACAO_ZEUS'),f.col('TIP_CURSO_ZEUS'),f.col('COD_SITUACAO_ACADEMICA'))
  
    return (df_acordos_atual
            #.drop(df_zeus['COD_RA_ZEUS'])   
            #.drop(df_acordos_atual['COD_PARCELAMENTO'])
           )

In [243]:
bi_zeus = base_zeus

In [244]:
df_acordos_atual = join_zeus_atual(df_acordos_promessas, bi_zeus)

In [248]:
def join_zeus_hist(df_acordos_promessas, bi_zeus, df_acordos_atual):  
    
    df_zeus = (bi_zeus.filter(f.col('TIP_ALUNO')== "ACORDO RAID")) 
    
    df_acordos_hist = (df_acordos_promessas.join(df_zeus,
                                  df_acordos_promessas['COD_PARCELAMENTO'] == df_zeus['COD_RA_ZEUS'], "inner"))
    
    df_acordos_hist = df_acordos_hist.select(f.col('COD_PARCELAMENTO'),f.col('STA_OPERACAO_ZEUS'),f.col('TIP_CURSO_ZEUS'),f.col('COD_SITUACAO_ACADEMICA'))
   
    df_total = df_acordos_hist.union(df_acordos_atual)
    
    df_acordos_promessas = (df_acordos_promessas.join(df_total,
                                  df_acordos_promessas['COD_PARCELAMENTO'] == df_total['COD_PARCELAMENTO'], "left"))
        
    win = Window.partitionBy(f.col("CHAVE")).orderBy(f.col("CHAVE").desc())  
    df_acordos_promessas = (df_acordos_promessas 
                          .withColumn("row", f.row_number().over(win))
                          .filter((f.col("row") == '1')))
    
    return (df_acordos_promessas
            .drop(df_total['COD_PARCELAMENTO'])   
            #.drop(df_acordos_atual['COD_RA_ZEUS'])
           )      

In [249]:
df2 = join_zeus_hist(df_acordos_promessas, bi_zeus, df_acordos_atual)

In [250]:
df2.count()

5313337

In [ ]:
df = df_acordos_promessas.select(f.col('COD_PARCELAMENTO'),f.col('COD_PARCELA'), f.concat(f.col('COD_PARCELAMENTO'),f.col('COD_PARCELA'))).limit(10)

In [ ]:
df.show()

In [ ]:
#dias atraso
df_bi_carteira_cobranca = (df_bi_carteira_cobranca.withColumn("CHAVE",f.concat(f.col('COD_PARCELAMENTO'),f.col('COD_PARCELA'))))

In [ ]:
df_bi_carteira_cobranca = df_bi_carteira_cobranca.dropDuplicates(['COD_CONTRATO'])

In [56]:
bi_zeus = base_zeus

In [57]:
df_acordos_promessas = join_zeus(df_acordos_promessas, bi_zeus)

In [ ]:
#--4� SIUTACAO ALUNO
df_acordos_promessas = (
        df_acordos_promessas
            .withColumn("SITUACAO_ALUNO", f.when(
                (f.year(df_acordos_promessas['DAT_CRIACAO_PARCELAMENTO']) == f.year(f.current_date())) &
                (f.month(df_acordos_promessas['DAT_CRIACAO_PARCELAMENTO']) == f.month(f.current_date())) 
            ,f.col('STA_OPERACAO_ZEUS')).otherwise(f.lit(None)))
    
            .withColumn("DSC_TIP_CURSO", f.when(
                (f.year(df_acordos_promessas['DAT_CRIACAO_PARCELAMENTO']) == f.year(f.current_date())) &
                (f.month(df_acordos_promessas['DAT_CRIACAO_PARCELAMENTO']) == f.month(f.current_date())) 
            ,f.col('DSC_TIPO_CURSO_ZEUS')).otherwise(f.lit(None)))
    
            .withColumn("TIP_CURSO", f.when(
                (f.year(df_acordos_promessas['DAT_CRIACAO_PARCELAMENTO']) == f.year(f.current_date())) &
                (f.month(df_acordos_promessas['DAT_CRIACAO_PARCELAMENTO']) == f.month(f.current_date())) 
            , f.col('TIP_CURSO_ZEUS')).otherwise(f.lit(None)))
    )

In [59]:
def join_vertia(df_acordos_promessas, bi_titulos_vertia):    
    df_acordos_promessas = (df_acordos_promessas.join(bi_titulos_vertia,
                                  f.ltrim(f.rtrim(df_acordos_promessas['COD_CONTRATO'])) == f.ltrim(f.rtrim(bi_titulos_vertia['COD_CONTRATO'])), "left"))
      
    df_acordos_promessas = (df_acordos_promessas.withColumn('CARTEIRA', f.lit('VERTIA')))

    
    return (df_acordos_promessas
            .drop(bi_titulos_vertia['COD_CONTRATO'])
            .drop(bi_titulos_vertia['COD_TITULO'])
            .drop(bi_titulos_vertia['DAT_PAGAMENTO'])
            .drop(bi_titulos_vertia['STA_TITULO'])
            .drop(bi_titulos_vertia['COD_CPFCGC_PES'])
           )

In [61]:
bi_titulos_vertia = titulos_vertia

In [62]:
df_acordos_promessas = join_vertia(df_acordos_promessas, bi_titulos_vertia) 

In [69]:
bi_polos_parceiros_e_proprios = polos_unidades

In [70]:
 # Polos
def join_polos(df_acordos_promessas, bi_polos_parceiros_e_proprios): 
    df_polos_parceiros_e_proprios = bi_polos_parceiros_e_proprios.select(f.col('TIP_POLO'),
                                                                   f.col('COD_CLBR'),
                                                                   f.col('COD_OLMP'),
                                                                   f.col('NOM_UNDE_SIST'),
                                                                   f.col('NOM_GRNT_NGCO'),
                                                                   f.col('NOM_GSRO_DRTR_UNDE'),
                                                                   f.col('NOM_GRPO_ECMC_GSRO'),
                                                                   f.col('NOM_GRPO_ECMC'),
                                                                   f.col('NOM_DRTR_RGNL'))
    
    df_acordos_promessas = (df_acordos_promessas.join(df_polos_parceiros_e_proprios,
                                  df_acordos_promessas['COD_FILIAL'] == df_polos_parceiros_e_proprios['COD_CLBR'], "left"))
   
    return df_acordos_promessas

In [71]:
df_acordos_promessas = join_polos(df_acordos_promessas, bi_polos_parceiros_e_proprios)  